# 🎯 Objetivos

El objetivo de este script es preparar los datos finales que servirán como entrada coherente y consistente para los distintos modelos del sistema de recomendación.

* ⚙️ **Configuración inicial**. Rutas, paquetes y funciones.
* ♻️ **Carga de datos**.
* 🧼 **Limpieza**. Tablas de datos finales.
* ✂️ **Train - Test**.
* ✅ **Chequeos finales**.
* 💾 **Guardado de información**. Guardado de información y resumen de los datos almacenados.

# ⚙️ Configuración

## Rutas (paths)

In [1]:
import os

# Rutas en función de la carpeta raíz del proyecto (README.md)
base_path = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if "__file__" in locals() else os.path.abspath("..")

# Subrutas
data_dir = os.path.join(base_path, "data")
db_path = os.path.join(data_dir, "steam.db")
src_dir = os.path.join(base_path, "src")
models_dir = os.path.join(base_path, "models")

## Paquetes y funciones

In [2]:
import warnings
warnings.filterwarnings("ignore")
import sqlite3
import pandas as pd

# ♻️ Carga de datos

In [3]:
# GAMES ESCALADA
conn = sqlite3.connect(db_path)
cur = conn.cursor()

games_esc = pd.read_sql("SELECT * FROM games_final_scaled", sqlite3.connect(db_path))

conn.close()

# Hacemos una copia para trabajar
games_esc = games_esc.copy()

In [4]:
# GAMES NO ESCALADA
conn = sqlite3.connect(db_path)
cur = conn.cursor()

games_no_esc = pd.read_sql("SELECT * FROM games_fe_clean_unscaled", sqlite3.connect(db_path))

conn.close()

# Hacemos una copia para trabajar
games_no_esc = games_no_esc.copy()

In [5]:
# USERS
conn = sqlite3.connect(db_path)
cur = conn.cursor()

user_games = pd.read_sql("SELECT * FROM user_games_filtered", sqlite3.connect(db_path))

conn.close()

# Hacemos una copia para trabajar
user_games = user_games.copy()

In [ ]:
# Asegurar consistencia
games_esc["appid"] = games_esc["appid"].astype(int)
games_no_esc["appid"] = games_no_esc["appid"].astype(int)

user_games["appid"] = user_games["appid"].astype(int)
user_games["user_id"] = user_games["user_id"].astype(int)

# 🧼 Limpieza

En este apartado se va a garantizar que todas las tablas están alineadas y contienen los mismos identificadores.

In [7]:
# Revisión de duplicados
assert games_esc["appid"].is_unique, "ERROR: appid duplicado en games_final_scaled"
assert games_no_esc["appid"].is_unique, "ERROR: appid duplicado en games_fe_clean_unscaled"

In [8]:
# Consistencia games_filtered
missing_from_games = set(games_no_esc["appid"]) - set(games_esc["appid"])
print(f"Juegos en games_no_esc que no están en games_esc: {len(missing_from_games)}")

if missing_from_games:
    print("Juegos faltantes:", list(missing_from_games)[:10])

# Comprobación de consistencia user_games
missing_from_games = set(user_games["appid"]) - set(games_esc["appid"])
print(f"Juegos en user_games que no están en games_esc: {len(missing_from_games)}")

Juegos en games_no_esc que no están en games_esc: 0
Juegos en user_games que no están en games_esc: 108


In [9]:
# Eliminamos esos juegos (sobreescribimos)
valid_games = set(games_esc["appid"])
games_no_esc_final = games_no_esc[games_no_esc["appid"].isin(valid_games)].copy()
user_games_final     = user_games[user_games["appid"].isin(valid_games)].copy()

In [10]:
# Comprobamos

# Comprobación de consistencia games_filtered
missing_from_games = set(games_no_esc_final["appid"]) - set(games_esc["appid"])
print(f"Juegos en games_no_esc_final que no están en games_esc: {len(missing_from_games)}")

# Comprobación de consistencia user_games
missing_from_games = set(user_games_final["appid"]) - set(games_esc["appid"])
print(f"Juegos en user_games_final que no están en games_esc: {len(missing_from_games)}")

Juegos en games_no_esc_final que no están en games_esc: 0
Juegos en user_games_final que no están en games_esc: 0


In [11]:
# Nos quedamos solo con los usuarios que han jugado a al menos 8 juegos (8 porque en el split necesitamos al menos 5 juegos en train)
games_per_user_total = (
    user_games_final.groupby("user_id")["appid"]
    .nunique()
)
valid_users = games_per_user_total[games_per_user_total >= 8].index
n_users_before = user_games_final["user_id"].nunique()
user_games_final = user_games_final[user_games_final["user_id"].isin(valid_users)].copy()
n_users_after = user_games_final["user_id"].nunique()
print(f"Usuarios eliminados por <8 juegos (antes del split): {n_users_before - n_users_after}")

Usuarios eliminados por <8 juegos (antes del split): 846


In [12]:
# Comprobamos si algun juego se ha quedo sin usuarios

all_appids_catalog = set(games_esc["appid"].unique())  
active_appids = set(user_games_final["appid"].unique())

# Juegos que se han quedado sin usuarios
orphan_games = all_appids_catalog - active_appids
print(f"Juegos huérfanos (sin usuarios tras filtrar): {len(orphan_games):,}")


Juegos huérfanos (sin usuarios tras filtrar): 0


In [13]:
# Número total de juegos únicos en cada tabla
n_games_no_esc_final = games_no_esc_final["appid"].nunique()
n_games_esc = games_esc["appid"].nunique()
n_user_games_final = user_games_final["appid"].nunique()

print(f"Juegos únicos en games_no_esc_final: {n_games_no_esc_final:,}")
print(f"Juegos únicos en games_esc: {n_games_esc:,}")
print(f"Juegos únicos en user_games_final: {n_user_games_final:,}")

Juegos únicos en games_no_esc_final: 2,468
Juegos únicos en games_esc: 2,468
Juegos únicos en user_games_final: 2,468


In [14]:
# Contar número de juegos únicos por usuario
games_per_user = (
    user_games_final.groupby("user_id")["appid"]
    .nunique()  # número de juegos jugados
    .reset_index(name="num_games")
)

# Media de juegos por usuario
mean_games = games_per_user["num_games"].mean()

# Resumen estadístico completo
summary = games_per_user["num_games"].describe(percentiles=[0.25, 0.5, 0.75, 0.9])

print(f"Media de juegos por usuario: {mean_games:.2f}")
print(summary)

Media de juegos por usuario: 146.78
count    47627.000000
mean       146.777059
std        169.566338
min          8.000000
25%         45.000000
50%         92.000000
75%        181.000000
90%        329.400000
max       2292.000000
Name: num_games, dtype: float64


# ✂️ Train - Test

El objetivo de este bloque es generar un conjunto de entrenamiento y otro de prueba por usuario.

## Leave-M-Out

In [15]:
# Barajamos las interacciones
user_games_shuffled = user_games_final.sample(frac=1, random_state=42)

# Enumeramos las interacciones por usuario
user_games_shuffled["rank"] = (
    user_games_shuffled
    .groupby("user_id")
    .cumcount()          
)

# Elegimos 3 interacciones aleatorias para el test (rank empieza en 0)
user_test_LMO = user_games_shuffled[user_games_shuffled["rank"] < 3].copy()
user_train_LMO = user_games_shuffled[user_games_shuffled["rank"] >= 3].copy()

user_test_LMO = user_test_LMO.drop(columns=["rank"])
user_train_LMO = user_train_LMO.drop(columns=["rank"])


# Filtramos juegos válidos
valid_appids = set(games_esc["appid"].tolist())

user_train_LMO = user_train_LMO[user_train_LMO["appid"].isin(valid_appids)]
user_test_LMO = user_test_LMO[user_test_LMO["appid"].isin(valid_appids)]

# ✅ Chequeos finales

In [16]:
# Usuarios presentes en train y test

train_users = set(user_train_LMO["user_id"].tolist())
test_users = set(user_test_LMO["user_id"].tolist())
common_users = train_users.intersection(test_users)

print("Usuarios en train:", len(train_users))
print("Usuarios en test :", len(test_users))
print("Usuarios comunes :", len(common_users))

# Nos quedamos solo con los usuarios que aparecen en ambos
user_train_LMO = user_train_LMO[user_train_LMO["user_id"].isin(common_users)]
user_test_LMO  = user_test_LMO[user_test_LMO["user_id"].isin(common_users)]

# Solapamiento exacto de interacciones (user_id, appid) entre train y test
overlap = pd.merge(
    user_train_LMO,
    user_test_LMO,
    on=["user_id", "appid"],
    how="inner"
)
print("Interacciones duplicadas en train y test:", len(overlap))
assert len(overlap) == 0, "Hay interacciones (user_id, appid) que aparecen en train y test"

Usuarios en train: 47627
Usuarios en test : 47627
Usuarios comunes : 47627
Interacciones duplicadas en train y test: 0


In [17]:
# Consistencia: interacciones de test deben estar en user_games_final
merged_test = user_test_LMO.merge(
    user_games_final,
    on=["user_id", "appid"],
    how="left",
    indicator=True
)
faltantes_test = (merged_test["_merge"] == "left_only").sum()
print("Interacciones de test no presentes en user_games_final:", faltantes_test)
assert faltantes_test == 0, "Hay interacciones en test que no están en user_games_final"

Interacciones de test no presentes en user_games_final: 0


In [18]:
# Duplicados user_games_final
dups = user_games_final.duplicated(subset=["user_id", "appid"]).sum()
print(f"Duplicados en user_games_final (user_id, appid): {dups}")
assert dups == 0, "Hay filas duplicadas (user_id, appid) en user_games_final"

Duplicados en user_games_final (user_id, appid): 0


In [19]:
# Al menos 5 juegos por usuario en train (post-split LMO)
train_counts = user_train_LMO.groupby("user_id")["appid"].nunique()
print("Usuarios con <5 juegos en train:", (train_counts < 5).sum())

Usuarios con <5 juegos en train: 0


In [20]:
# Distribución de interacciones y densidad de la matriz
print("Usuarios únicos en user_games_final:", user_games_final["user_id"].nunique())
print("Juegos únicos en user_games_final:", user_games_final["appid"].nunique())
print("Total interacciones en user_games_final:", len(user_games_final))

densidad = len(user_games_final) / (
    user_games_final["user_id"].nunique() * user_games_final["appid"].nunique()
)
print("Densidad matriz user_games_final:", densidad)

Usuarios únicos en user_games_final: 47627
Juegos únicos en user_games_final: 2468
Total interacciones en user_games_final: 6990551
Densidad matriz user_games_final: 0.05947206613902285


In [21]:
# Cobertura de juegos entre train/test y catálogos finales
apps_train = set(user_train_LMO["appid"])
apps_test  = set(user_test_LMO["appid"])
apps_eval  = apps_train | apps_test   # unión

apps_scaled    = set(games_esc["appid"])
apps_noscaled  = set(games_no_esc_final["appid"])
apps_usergames = set(user_games_final["appid"])

# Juegos de train+test que faltan en cada tabla final
missing_in_scaled    = apps_eval - apps_scaled
missing_in_noscaled  = apps_eval - apps_noscaled
missing_in_usergames = apps_eval - apps_usergames

print("Total juegos en train+test:", len(apps_eval))
print("Faltan en games_escaled_final      :", len(missing_in_scaled))
print("Faltan en games_no_escaled_final   :", len(missing_in_noscaled))
print("Faltan en user_games_final (por appid):", len(missing_in_usergames))

# Juegos de user_games_final que no están en games_escaled_final
extra_in_usergames = apps_usergames - apps_scaled
print("Juegos en user_games_final fuera de games_escaled_final:", len(extra_in_usergames))
assert len(extra_in_usergames) == 0, "Hay appid en user_games_final que no están en games_escaled_final"

Total juegos en train+test: 2468
Faltan en games_escaled_final      : 0
Faltan en games_no_escaled_final   : 0
Faltan en user_games_final (por appid): 0
Juegos en user_games_final fuera de games_escaled_final: 0


# 💾 Guardar información

In [22]:
# Guardamos las versiones finales
conn = sqlite3.connect(db_path)
games_esc.to_sql("games_escaled_final", conn, if_exists="replace", index=False)
games_no_esc_final.to_sql("games_no_escaled_final", conn, if_exists="replace", index=False)
user_games_final.to_sql("user_games_final", conn, if_exists="replace", index=False)
conn.close()

# Número de filas guardadas
print(f"Filas en games_escaled_final: {len(games_esc)}")
print(f"Filas en games_no_escaled_final: {len(games_no_esc_final)}")
print(f"Filas en user_games_final: {len(user_games_final)}")

# Usuarios y juegos en user_games_final
print(f"Usuarios únicos en user_games_final: {user_games_final["user_id"].nunique()}")
print(f"Juegos  únicos en user_games_final: {user_games_final["appid"].nunique()}")

Filas en games_escaled_final: 2468
Filas en games_no_escaled_final: 2468
Filas en user_games_final: 6990551
Usuarios únicos en user_games_final: 47627
Juegos  únicos en user_games_final: 2468


In [23]:
# Guardamos la división para poder reutilizarla
conn = sqlite3.connect(db_path)
user_train_LMO.to_sql("user_train_LMO", conn, if_exists="replace", index=False)
user_test_LMO.to_sql("user_test_LMO", conn, if_exists="replace", index=False)
conn.close()

# Número de filas guardadas
print(f"Filas en user_train: {len(user_train_LMO)}")
print(f"Filas en user_test: {len(user_test_LMO)}")

# Usuarios y juegos en train y test

print(f"Usuarios únicos en user_train_LMO: {user_train_LMO["user_id"].nunique()}")
print(f"Juegos únicos en user_train_LMO: {user_train_LMO["appid"].nunique()}")
print(f"Usuarios únicos en user_test_LMO: {user_test_LMO["user_id"].nunique()}")
print(f"Juegos únicos en user_test_LMO: {user_test_LMO["appid"].nunique()}")

Filas en user_train: 6847670
Filas en user_test: 142881
Usuarios únicos en user_train_LMO: 47627
Juegos únicos en user_train_LMO: 2468
Usuarios únicos en user_test_LMO: 47627
Juegos únicos en user_test_LMO: 2460
